In [48]:
# Import dependencies
import numpy as np
import pandas as pd

In [49]:
# Load in the dataset
df = pd.read_csv('../../datasets/heart.csv')

X = df.drop(columns=['target']).to_numpy()
y = df['target'].to_numpy().reshape((303,1)) # Force the 1 dimension

# Perform Z-score normalization on x
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

In [50]:
# Split into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, shuffle=True)

In [51]:
def HingeLoss(weights: np.ndarray, y: np.ndarray, y_hat: np.ndarray, C = 1.0):
    loss_hinge = np.maximum(0, 1 - y * y_hat)
    return (1/2) * (np.dot(weights.T, weights)) + C * np.sum(loss_hinge)

In [52]:
# Hyperparameters
learning_rate = 0.0005
epochs = 1000

In [53]:
class LinearSVM:
    def __init__(self, num_features, learning_rate):
        self.lr = learning_rate
        self.w = np.zeros((num_features, 1))
        self.b = 0

        # For adam gradient descent
        self.w_m = 0
        self.w_v = 0
        self.b_m = 0
        self.b_v = 0
        self.beta1 = 0.9
        self.beta2 = 0.999

    def forward(self, X: np.ndarray):
        return (X @ self.w) + self.b
    
    def predict(self, X: np.ndarray):
        return (self.forward(X) >= 0).astype(int)

    def update_weights(self, X: np.ndarray, y: np.ndarray, iteration: int, C = 1.0):
        n = len(y) # Get length of examples

        # Get predictions
        y_hat = self.forward(X)
        condition = y * y_hat >= 1

        # Extract only the few gradients that have violated the margins
        y_reshaped = y[:, np.newaxis] # make into (n, 1)
        hinge_grad_w = np.where(condition[:, np.newaxis], 0, -C * X.T @ y)

        dw = self.w + np.mean(hinge_grad_w, axis=0)
        db = np.mean(np.where(condition, 0, -C * y), axis=0)

        # Adam momentums
        self.w_m = self.beta1 * self.w_m + (1 - self.beta1) * dw # SGD + Momentum
        self.w_v = self.beta2 * self.w_v + (1 - self.beta2) * dw * dw # RMS Prop

        self.b_m = self.beta1 * self.b_m + (1 - self.beta1) * db # SGD + Momentum
        self.b_v = self.beta2 * self.b_v + (1 - self.beta2) * db * db # RMS Prop

        # Unbias them
        self.w_m_hat = self.w_m / (1 - self.beta1 ** iteration)
        self.w_v_hat = self.w_v / (1 - self.beta2 ** iteration)

        self.b_m_hat = self.b_m / (1 - self.beta1 ** iteration)
        self.b_v_hat = self.b_v / (1 - self.beta2 ** iteration)

        # Apply GD
        self.w -= (self.lr * self.w_m_hat) / (np.sqrt(self.w_v_hat) + 1e-7)
        self.b -= (self.lr * self.b_m_hat) / (np.sqrt(self.b_v_hat) + 1e-7)

    # Fit the data onto the logistic regression function
    def fit(self, X: np.ndarray, y: np.ndarray, epochs=1000, verbose=False):
        for i in range(epochs):
            # Perform gradient descent
            self.update_weights(X, y, i + 1) # plus 1 to avoid divide by zero

            # Report loss if verbose
            if verbose and i % 100 == 0:
                loss = HingeLoss(self.w, y, self.forward(X))
                print(f'[{i}]: Loss: {loss}')


In [65]:
model = LinearSVM(X.shape[1], learning_rate=learning_rate)
model.fit(X_train, y_train, 10000, True)

[0]: Loss: [[256.68307124]]
[100]: Loss: [[225.01222282]]
[200]: Loss: [[194.94847588]]
[300]: Loss: [[177.79417196]]
[400]: Loss: [[170.61275572]]
[500]: Loss: [[166.82799604]]
[600]: Loss: [[165.31013921]]
[700]: Loss: [[164.7451952]]
[800]: Loss: [[164.42051125]]
[900]: Loss: [[164.62394076]]
[1000]: Loss: [[165.18772821]]
[1100]: Loss: [[165.98674093]]
[1200]: Loss: [[167.01856953]]
[1300]: Loss: [[168.26474585]]
[1400]: Loss: [[169.59663042]]
[1500]: Loss: [[170.99780576]]
[1600]: Loss: [[172.54269841]]
[1700]: Loss: [[174.23635191]]
[1800]: Loss: [[175.95763329]]
[1900]: Loss: [[177.75248108]]
[2000]: Loss: [[179.64597551]]
[2100]: Loss: [[181.64027005]]
[2200]: Loss: [[183.65420966]]
[2300]: Loss: [[185.6860377]]
[2400]: Loss: [[187.77830442]]
[2500]: Loss: [[189.89381653]]
[2600]: Loss: [[192.06179118]]
[2700]: Loss: [[194.26375497]]
[2800]: Loss: [[196.48252889]]
[2900]: Loss: [[198.72586435]]
[3000]: Loss: [[201.02552065]]
[3100]: Loss: [[203.34201194]]
[3200]: Loss: [[205.67

In [66]:
# Evaluate model
def eval(X: np.ndarray, y: np.ndarray, model: LinearSVM):
    # Get predictions (from actual labels)
    y_hat = model.predict(X)

    # Calculate accruacy
    accuracy = np.mean(y_hat == y)
    print(f'Test accuracy: {accuracy:.3f}')

eval(X_test, y_test, model)

Test accuracy: 0.848
